# Assignment 1 — Gradio dashboard

Run with `uv run python main.py app` (after `main.py train`). Everything here is loaded from `artifacts/`,
which the training stage wrote: the fitted scaler, the three saved models and the feature table.
**Nothing is trained here**; the models are only used to predict.

Tabs:
1. **Model comparison**: metrics and predicted-vs-actual (calibration) plot for the three models and the baselines.
2. **Distributions**: feature and target distributions.
3. **Betting threshold**: move `t` and watch the confusion matrix and the profit change.
4. **Match explorer**: the three models' probability for a single match, next to the market's.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, brier_score_loss, confusion_matrix, log_loss, roc_auc_score

ART_DIR = Path.cwd() / "artifacts"
if not (ART_DIR / "config.json").exists():
    raise SystemExit("artifacts/ not found: run `uv run python main.py train` first.")

config = json.loads((ART_DIR / "config.json").read_text())
FEATURES = config["features"]
T_DEFAULT = config["t_final"]
TARGET = "home_win"
ODDS_PRE = ["Pinnacle home win odds", "Pinnacle draw odds", "Pinnacle away win odds"]

df = pd.read_csv(ART_DIR / "features.csv", parse_dates=["Match Date"])

## Load the three saved models and predict

Same scaler for all three, fitted on train during training.

In [ ]:
scaler = joblib.load(ART_DIR / "scaler.joblib")
sk_model = joblib.load(ART_DIR / "model_sklearn.joblib")

manual = torch.load(ART_DIR / "model_torch_manual.pt")

std_model = nn.Sequential(nn.Linear(len(FEATURES), 1), nn.Sigmoid())   # same architecture as in training
std_model.load_state_dict(torch.load(ART_DIR / "model_torch_standard.pt"))
std_model.eval()

X = scaler.transform(df[FEATURES])
X_t = torch.tensor(X, dtype=torch.float32)
with torch.no_grad():
    df["p_sklearn"] = sk_model.predict_proba(X)[:, 1]
    df["p_torch_manual"] = torch.sigmoid(X_t @ manual["w"] + manual["b"]).numpy()
    df["p_torch_standard"] = std_model(X_t).squeeze(-1).numpy()

has_odds = df[ODDS_PRE].notna().all(axis=1)
inv = 1 / df[ODDS_PRE]
df["p_pinnacle"] = np.where(has_odds, inv.iloc[:, 0] / inv.sum(axis=1), np.nan)   # margin removed
df["p_constant"] = config["train_rate"]

MODELS = {"sklearn": "p_sklearn", "PyTorch manual": "p_torch_manual", "PyTorch standard": "p_torch_standard"}
BASELINES = {"Pinnacle (market)": "p_pinnacle", "Constant (train rate)": "p_constant"}
SPLITS = ["val", "test", "train"]

## Tab 1: model comparison

In [ ]:
def metrics_table(split):
    d = df[df["split"] == split]
    rows = []
    for name, col in {**MODELS, **BASELINES}.items():
        dd = d.dropna(subset=[col])
        y, p = dd[TARGET].values, dd[col].values
        rows.append({"model": name, "n": len(dd), "log loss": log_loss(y, p), "Brier": brier_score_loss(y, p),
                     "AUC": roc_auc_score(y, p) if np.unique(p).size > 1 else np.nan,
                     "accuracy (p≥0.5)": accuracy_score(y, (p >= 0.5).astype(int))})
    return pd.DataFrame(rows).round(4)

def calibration_plot(split, n_bins=10):
    """Predicted vs actual: matches grouped by predicted probability, against the realised home-win rate."""
    d = df[df["split"] == split]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                             line=dict(dash="dash", color="grey")))
    styles = {"sklearn": dict(width=6), "PyTorch manual": dict(width=3, dash="dot"),
              "PyTorch standard": dict(width=2, dash="dash"), "Pinnacle (market)": dict(width=2)}
    for name, col in {**MODELS, "Pinnacle (market)": "p_pinnacle"}.items():
        dd = d.dropna(subset=[col])
        bins = pd.qcut(dd[col], n_bins, duplicates="drop")
        g = dd.groupby(bins, observed=True).agg(pred=(col, "mean"), actual=(TARGET, "mean"), n=(TARGET, "size"))
        fig.add_trace(go.Scatter(x=g["pred"], y=g["actual"], mode="lines+markers", name=name,
                                 line=styles[name], customdata=g["n"],
                                 hovertemplate="predicted %{x:.3f}<br>actual %{y:.3f}<br>n=%{customdata}"))
    fig.update_layout(title=f"Predicted vs actual home-win rate ({split}, {n_bins} quantile bins)",
                      xaxis_title="mean predicted P(home win)", yaxis_title="realised home-win rate",
                      xaxis_range=[0, 1], yaxis_range=[0, 1], height=500)
    return fig

def agreement_plot(split):
    """Each match's probability from the two PyTorch models against sklearn: all points sit on the diagonal."""
    d = df[df["split"] == split]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d["p_sklearn"], y=d["p_torch_manual"], mode="markers", name="PyTorch manual",
                             marker=dict(size=5, opacity=0.5)))
    fig.add_trace(go.Scatter(x=d["p_sklearn"], y=d["p_torch_standard"], mode="markers", name="PyTorch standard",
                             marker=dict(size=5, opacity=0.5, symbol="x")))
    gap = max((d["p_torch_manual"] - d["p_sklearn"]).abs().max(), (d["p_torch_standard"] - d["p_sklearn"]).abs().max())
    fig.update_layout(title=f"Per-match agreement with sklearn ({split}) — max |difference| = {gap:.5f}",
                      xaxis_title="sklearn P(home win)", yaxis_title="PyTorch P(home win)", height=450)
    return fig

coefficients = pd.read_csv(ART_DIR / "coefficients.csv", index_col=0).round(4).reset_index(names="feature")

def update_comparison(split):
    return metrics_table(split), calibration_plot(split), agreement_plot(split)

## Tab 2: distributions

In [ ]:
ALL_FEATURES = [f for g in config["feature_groups"].values() for f in g]

def feature_distribution(feature, split):
    d = df[df["split"] == split] if split != "all" else df
    d = d.assign(outcome=np.where(d[TARGET] == 1, "home win", "draw / away win"))
    used = "used by the models" if feature in FEATURES else "dropped after ablation"
    fig = px.histogram(d, x=feature, color="outcome", barmode="overlay", histnorm="probability density",
                       nbins=40, opacity=0.6,
                       title=f"{feature} ({used}) by outcome — {split}")
    fig.update_layout(height=450)
    return fig

def target_distribution():
    g = df.groupby(["Season", "League Division"])[TARGET].mean().reset_index()
    fig = px.line(g, x="Season", y=TARGET, color="League Division", markers=True,
                  title="Home-win rate by season and division (the target)")
    fig.update_layout(yaxis_tickformat=".0%", yaxis_title="home-win rate", height=400)
    return fig

## Tab 3: betting threshold

Rule from REPORT.md: back the home team when `p_model − p_Pinnacle ≥ t`, 1 € at the pre-closing Pinnacle odds.
"Bet" is the positive prediction in the confusion matrix. Matches without odds are never bet on.

In [ ]:
def bets_for(split, model, t):
    d = df[(df["split"] == split)].copy()
    d["bet"] = d["p_pinnacle"].notna() & ((d[MODELS[model]] - d["p_pinnacle"]) >= t)
    won = d[TARGET] == 1
    d["profit"] = np.where(d["bet"] & won, d["Pinnacle home win odds"] - 1, 0.0) + np.where(d["bet"] & ~won, -1.0, 0.0)
    return d

def confusion_fig(d, t):
    cm = confusion_matrix(d[TARGET], d["bet"].astype(int), labels=[1, 0])
    labels = [["bet & home win<br>(win odds − 1)", "no bet & home win<br>(missed, 0 €)"],
              ["bet & no home win<br>(lose 1 €)", "no bet & no home win<br>(avoided, 0 €)"]]
    text = [[f"{labels[i][j]}<br><b>{cm[i, j]}</b>" for j in range(2)] for i in range(2)]
    fig = go.Figure(go.Heatmap(z=cm, x=["bet", "no bet"], y=["home win", "no home win"], text=text,
                               texttemplate="%{text}", colorscale="Blues", showscale=False))
    fig.update_layout(title=f"Confusion matrix at t = {t:+.2f}", yaxis_autorange="reversed", height=400)
    return fig

def threshold_curve(split, model, t):
    ts = np.round(np.arange(-0.10, 0.205, 0.01), 2)
    rows = []
    for tt in ts:
        d = bets_for(split, model, tt)
        n = int(d["bet"].sum())
        rows.append({"t": tt, "profit": d["profit"].sum(), "n_bets": n})
    r = pd.DataFrame(rows)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=r["t"], y=r["profit"], name="profit (€)", customdata=r["n_bets"],
                         hovertemplate="t=%{x:+.2f}<br>profit %{y:.2f} €<br>%{customdata} bets"))
    fig.add_vline(x=t, line_dash="dash", line_color="red", annotation_text=f"t = {t:+.2f}")
    fig.update_layout(title="Total profit for every threshold", xaxis_title="t", yaxis_title="profit (€)", height=380)
    return fig

def cumulative_fig(d, t):
    d = d.sort_values("Match Date")
    fig = go.Figure(go.Scatter(x=d["Match Date"], y=d["profit"].cumsum(), mode="lines", name="cumulative profit"))
    fig.add_hline(y=0, line_color="grey")
    fig.update_layout(title=f"Cumulative profit through the season at t = {t:+.2f}",
                      xaxis_title="date", yaxis_title="€", height=380)
    return fig

def update_threshold(split, model, t):
    d = bets_for(split, model, t)
    n = int(d["bet"].sum())
    profit = d["profit"].sum()
    hit = d.loc[d["bet"], TARGET].mean() if n else float("nan")
    roi = profit / n if n else float("nan")
    expected = d.loc[d["bet"], "p_pinnacle"].mean() if n else float("nan")
    summary = (f"### {model} on {split} at t = {t:+.2f}\n"
               f"| bets placed | hit rate | market expected | profit | ROI |\n|---|---|---|---|---|\n"
               f"| {n} of {len(d)} | {hit:.1%} | {expected:.1%} | **{profit:+.2f} €** | **{roi:+.1%}** |\n\n"
               f"*Default t = {T_DEFAULT:+.2f} was chosen on validation. "
               f"'Market expected' is Pinnacle's average probability for the matches bet on.*")
    return summary, confusion_fig(d, t), threshold_curve(split, model, t), cumulative_fig(d, t)

## Tab 4: match explorer

In [ ]:
test_matches = df[df["split"] == "test"].sort_values("Match Date")
match_labels = [f"{r['Match Date']:%Y-%m-%d} · {r['Home Team']} vs {r['Away Team']}" for _, r in test_matches.iterrows()]
label_to_idx = dict(zip(match_labels, test_matches.index))

def explore_match(label):
    r = df.loc[label_to_idx[label]]
    probs = {name: r[col] for name, col in MODELS.items()}
    probs["Pinnacle (market)"] = r["p_pinnacle"]
    fig = go.Figure(go.Bar(x=list(probs), y=list(probs.values()), text=[f"{v:.1%}" for v in probs.values()],
                           textposition="outside"))
    fig.update_layout(title=f"P(home win): {r['Home Team']} vs {r['Away Team']}", yaxis_range=[0, 1],
                      yaxis_tickformat=".0%", height=400)
    result = {"H": "home win", "D": "draw", "A": "away win"}[r["Full Time Result (H=Home Win, D=Draw, A=Away Win)"]]
    edge = r["p_sklearn"] - r["p_pinnacle"]
    info = (f"**Result:** {result} · **Pinnacle home odds:** {r['Pinnacle home win odds']:.2f} · "
            f"**Edge (model − market):** {edge:+.3f} → {'bet' if edge >= T_DEFAULT else 'no bet'} at t = {T_DEFAULT:+.2f}")
    feats = pd.DataFrame({"feature": FEATURES, "value": [round(r[f], 3) for f in FEATURES]})
    return fig, info, feats

## Layout and launch

In [ ]:
with gr.Blocks(title="LaLiga home-win model") as demo:
    gr.Markdown("# Should we back the home team?\n"
                "Logistic regression trained three ways (scikit-learn, manual PyTorch, `nn.Module` + `torch.optim`) "
                "on LaLiga and Segunda 2014-15 → 2022-23; validated on 2023-24, tested on 2024-25. "
                "All models are loaded from `artifacts/`; nothing is retrained.")

    with gr.Tab("Model comparison"):
        split_cmp = gr.Radio(SPLITS, value="test", label="Split")
        table_cmp = gr.Dataframe(label="Metrics (lower log loss / Brier is better)")
        with gr.Row():
            plot_cal = gr.Plot()
            plot_agree = gr.Plot()
        gr.Dataframe(value=coefficients, label="Coefficients on standardised features (from training)")
        split_cmp.change(update_comparison, split_cmp, [table_cmp, plot_cal, plot_agree])

    with gr.Tab("Distributions"):
        with gr.Row():
            feat_dd = gr.Dropdown(ALL_FEATURES, value="diff_elo", label="Feature")
            split_dist = gr.Radio(["all", "train", "val", "test"], value="all", label="Split")
        plot_feat = gr.Plot()
        plot_target = gr.Plot(value=target_distribution())
        feat_dd.change(feature_distribution, [feat_dd, split_dist], plot_feat)
        split_dist.change(feature_distribution, [feat_dd, split_dist], plot_feat)

    with gr.Tab("Betting threshold"):
        with gr.Row():
            split_bet = gr.Radio(["val", "test"], value="test", label="Split")
            model_bet = gr.Dropdown(list(MODELS), value="sklearn", label="Model")
            t_slider = gr.Slider(-0.10, 0.20, value=T_DEFAULT, step=0.01, label="t: required edge over the market")
        summary_md = gr.Markdown()
        with gr.Row():
            plot_cm = gr.Plot()
            plot_curve = gr.Plot()
        plot_cum = gr.Plot()
        inputs = [split_bet, model_bet, t_slider]
        outputs = [summary_md, plot_cm, plot_curve, plot_cum]
        for comp in inputs:
            comp.change(update_threshold, inputs, outputs)

    with gr.Tab("Match explorer"):
        match_dd = gr.Dropdown(match_labels, value=match_labels[0], label="Test-season match")
        plot_match = gr.Plot()
        info_md = gr.Markdown()
        feats_tbl = gr.Dataframe(label="Features for this match (before scaling)")
        match_dd.change(explore_match, match_dd, [plot_match, info_md, feats_tbl])

    demo.load(update_comparison, split_cmp, [table_cmp, plot_cal, plot_agree])
    demo.load(feature_distribution, [feat_dd, split_dist], plot_feat)
    demo.load(update_threshold, inputs, outputs)
    demo.load(explore_match, match_dd, [plot_match, info_md, feats_tbl])

demo.launch()